# 🎯 [MNT10_M07] 긴급 출동 순서 변경 및 정비반 재배치를 위한 정비대대장 HITL 승인 체계 설계
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MNT10/mnt10_m07_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 장비정비·수리부속 (Level 3) |
| **미션 ID** | MNT10_M07 |
| **미션 주제** | 긴급 출동 순서 변경 및 정비반 재배치를 위한 정비대대장 HITL 승인 체계 설계 |
| **소요 시간** | 약 3.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
야전에서 장비가 고장 나면 이동정비반이 얼마나 빨리 도착해 고치느냐가 부대 전투력 회복 속도를 좌우하지만, 가까운 정비반부터 보내는 방식은 요청이 몰리는 지역의 정비반에 부하를 집중시키고 핵심 장비를 뒤로 미루는 문제가 있다. 본 프로젝트는 정비인력의 부하율과 공정 균형을 데이터로 예측·관리하는 것을 목표로, 교육용 가상 지역의 긴급 정비 요청과 이동정비반 자료를 코드로 만들어 분석한다. 요청 데이터 스키마 정의, 정비반 상태 자료 정제, 거리 기준 Baseline과 탐색적 분석, 중요도·정비 소요시간·이동 경로를 종합한 우선순위 엔진, 출동 지연 위험도와 현장 수리 가능성 점수, 관제 대시보드와 설명가능성, 정비대대장 승인(HITL) 시뮬레이터, 동시 다발 요청 스트레스 시험, AAR 가중치 재튜닝과 효과 산출, 재학습을 포함한 통합 도구까지 단계별로 구현한다. 최종 목표는 핵심 장비의 가동중단 시간을 줄이면서 정비반 간 부하를 고르게 하고, 출동 순서 변경과 재배치는 지휘관이 근거를 보고 결정하도록 지원하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

### 📁 프로젝트 산출물(deliverables) 디렉토리 구성 안내

본 과제는 실무 현장에서 바로 쓰일 수 있는 국방 소프트웨어 아키텍처 표준을 지향합니다.  
과제를 진행하며 생성되는 산출물은 아래 규칙에 따라 각 폴더에 체계적으로 저장해 주세요.

| 디렉토리 | 군수 실무 비유 | 주 역할 및 산출물 파일 | 분리 목적 |
| :--- | :--- | :--- | :--- |
| **`docs/`** | 야전 교범 / 정비 지침서 | `mnt10_logistics_domain_spec.md` (미션1), `mnt10_operations_guide.md` (미션10) | 코드를 모르는 지휘관/후임자도 데이터 규격과 비상 매뉴얼을 이해하도록 보장 |
| **`notebooks/`** | 정비 시험실 / 연구실 | `mnt10_eda_analysis.ipynb` (미션3), 본 실습 노트북 | AI와 대화하며 자유롭게 데이터를 시각화하고 알고리즘을 반복 실험 |
| **`src/`** | 실제 장착 핵심 부품 모듈 | `src/etl/`, `src/features/`, `src/models/`, `src/dashboard/`, `src/optimization/` | 운영 서버에서 자동 배치 실행 가능한 순수 파이썬 모듈(`.py`)로 부품화 |
| **`reports/`** | 지휘관 결심 보고서 | `mnt10_roi_analysis.md` (미션9), AAR 오배정 분석서 | 정비대대장/군수참모에게 정비 지연 15% 감축 등 정량 ROI 성과 보고 |
| **`pipeline/`** | 자동화 컨베이어 벨트 | `pipeline/mnt10_retrain.py` (미션10) | 긴급 요청 유입 시 전처리부터 관제까지 자동 구동 및 재학습 파이프라인 |

> **💡 [바이브 코딩(AI 협업) 실습 요령]**  
> 1. **노트북에서 먼저 검증**: 본 노트북의 각 미션 셀에서 AI에게 질의하며 단계별 코드를 실행하고 결과를 확인합니다.  
> 2. **AI에게 모듈화 요청**: 검증 완료 후 AI에게 *"이 코드를 `src/.../` 파일에 넣을 수 있게 함수/클래스로 모듈화해줘"*라고 요청합니다.  
> 3. **디렉토리별 저장**: 제안된 코드를 해당 파일에 저장하여 자연스럽게 실무 패키지를 완성합니다.

---



In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 07. 긴급 출동 순서 변경 및 정비반 재배치를 위한 정비대대장 HITL 승인 체계 설계
**소요시간**: 약 3.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

출동 순서 변경을 위한 정비대대장 HITL 승인 체계를 설계한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **긴급 출동 순서 변경과 정비반 재배치를 정수계획(할당 문제)으로 제안하고, 정비대대장 승인을 거쳐야만 적용되는 HITL 승인 체계를 시뮬레이터로 설계한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **배정 주기마다 기본 계획(접수 순서·최근접)과 할당 최적화 제안을 비교해, 다르고 기대 이득이 있으면 승인 대기열에 올린다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/optimization/mnt10_simulator.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 현실적 군수 제약조건 반영도 및 시나리오별 대안 제시 적절성

#### 미션 PASS 조건

- 정량 기준: 최적화 적용 시 비용 15% 절감 또는 재고 회전율 20% 향상


- 진행 기준: 선형/정수 계획법(LP/MIP) 최적화 해 도출 및 시뮬레이션 완료


- 검증 방법: 과거 실적 데이터 대비 최적화 결과 비교 백테스팅


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""긴급 출동 순서 변경과 정비반 재배치를 정수계획(할당 문제)으로 제안하되, 기본 계획(접수 순서·최근접 정비반)과 다르면 정비대대장 승인 대기열에 올려 승인·반려·무응답(기본 계획 유지)을 사유와 함께 해시 체인 감사 로그에 남기는 HITL 시뮬레이터를 구현한다. 기본 계획 대비 가동중단 비용 절감률과 승인 지연의 영향을 비교한다."""
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

# ---------------------------------------------------------------------------
# [교육용가상자료] 야전 이동정비반·긴급 정비 요청 생성기 (실제 부대·위치·장비·인원 정보 없음)
# 60km x 60km 가상 지역에 이동정비반 6개와 30일간의 긴급 정비 요청을 만든다.
# 장비 구분(mobility/fires/comms/power)과 고장 유형은 교육용 추상 분류다.
# ---------------------------------------------------------------------------
AREA_KM, DUTY_H = 60, 10.0                     # 가상 작전지역 크기, 정비반 1일 기준 가용시간
FAULT_H = {"engine": 6.0, "drivetrain": 5.0, "hydraulic": 3.0, "electric": 2.0, "comm_module": 1.5, "cooling": 2.5}
FIELD_P = {"engine": 0.3, "drivetrain": 0.35, "hydraulic": 0.75, "electric": 0.9, "comm_module": 0.95, "cooling": 0.8}
EQUIP_F = {"mobility": 1.0, "fires": 1.2, "comms": 0.7, "power": 0.8}
FAULT_OF = {"mobility": [0.3, 0.25, 0.2, 0.1, 0.0, 0.15], "fires": [0.2, 0.15, 0.35, 0.15, 0.0, 0.15],
            "comms": [0.0, 0.0, 0.0, 0.4, 0.5, 0.1], "power": [0.35, 0.0, 0.0, 0.35, 0.0, 0.3]}
CRIT_P = {"mobility": [0.45, 0.35, 0.2], "fires": [0.2, 0.35, 0.45], "comms": [0.3, 0.4, 0.3], "power": [0.6, 0.3, 0.1]}
HUBS = [(15, 20), (45, 25), (30, 50)]           # 가상 훈련장 3곳(요청이 몰리는 지점)


def make_teams(seed=0):
    rng = np.random.default_rng(seed)
    base = [(8, 10), (30, 6), (52, 12), (10, 44), (32, 38), (54, 50)]
    skills = [["fires", "mobility"], ["comms", "mobility"], ["fires", "power"],
              ["mobility", "power"], ["comms", "fires"], ["comms", "mobility", "power"]]
    return [{"team": f"MT{i + 1}", "x": float(x), "y": float(y), "crew": int(rng.integers(3, 6)), "skills": s}
            for i, ((x, y), s) in enumerate(zip(base, skills))]


def make_requests(n_days=30, per_day=10, seed=1, surge_days=(8, 9, 10, 11, 22, 23, 24)):
    """하루 평균 per_day건, 훈련 주간(surge_days)에는 1.6배의 긴급 정비 요청을 만든다."""
    rng = np.random.default_rng(seed)
    rows = []
    for d in range(n_days):
        n = rng.poisson(per_day * (1.6 if d in surge_days else 1.0))
        weather = rng.choice(["clear", "rain", "snow"], p=[0.65, 0.25, 0.1])
        for _ in range(n):
            if rng.random() < 0.7:
                hx, hy = HUBS[rng.integers(0, 3)]
                x, y = np.clip(rng.normal(hx, 6), 0, AREA_KM), np.clip(rng.normal(hy, 6), 0, AREA_KM)
            else:
                x, y = rng.uniform(0, AREA_KM, 2)
            eq = str(rng.choice(list(EQUIP_F), p=[0.45, 0.25, 0.2, 0.1]))
            fault = str(rng.choice(list(FAULT_H), p=FAULT_OF[eq]))
            parts = bool(rng.random() < 0.75)
            bad = weather != "clear"
            std_h = FAULT_H[fault] * EQUIP_F[eq] * (1.35 if not parts else 1.0) * (1.15 if bad else 1.0) * rng.lognormal(0, 0.12)
            rows.append({"t_req": d * 24 + float(np.clip(rng.normal(13, 4), 5, 23)), "day": d,
                         "x": float(x), "y": float(y), "equip": eq, "fault": fault,
                         "crit": int(rng.choice([1, 2, 3], p=CRIT_P[eq])), "parts_ready": parts,
                         "weather": str(weather), "road": "unpaved" if y > 30 and rng.random() < 0.7 else "paved",
                         "field_ok": bool(rng.random() < FIELD_P[fault] * (1.0 if parts else 0.5)),
                         "std_repair_h": round(float(std_h), 2), "road_delay": float(rng.lognormal(0, 0.15))})
    df = pd.DataFrame(rows).sort_values("t_req").reset_index(drop=True)
    df.insert(0, "req_id", [f"R{i:04d}" for i in range(len(df))])
    return df
# ---------------------------------------------------------------------------

# ---- 출동 배정·정비 모의 공통 함수 ----
W0 = {"crit": 1.0, "wait": 0.3, "short": 0.2, "balance": 0.5}   # 우선순위·배정 기본 가중치(미션 9에서 재튜닝)


def travel_h(team, r, actual=False):
    """도로 굴곡 1.3배, 포장 35km/h·비포장 25km/h, 악천후 20% 감속. actual=True면 실제 도로 지연을 반영."""
    d = 1.3 * np.hypot(team["x"] - r["x"], team["y"] - r["y"])
    v = (35.0 if r["road"] == "paved" else 25.0) / (1.2 if r["weather"] != "clear" else 1.0)
    return d / v * (r["road_delay"] if actual else 1.0)


def on_site_h(team, r):
    """현장 정비시간: 4인 기준 표준시간을 투입 인원으로 환산. 현장 수리 불가면 진단·후송 조치 1시간."""
    return r["std_repair_h"] * 4 / team["crew"] if r["field_ok"] else 1.0


def priority(r, now, est_h, w):
    return w["crit"] * r["crit"] + w["wait"] * (now - r["t_req"]) + w["short"] / (1 + est_h)


def simulate(req, teams, policy="baseline", w=None, cycle_h=1.0, est=None):
    """배정 주기(cycle_h)마다 대기 요청을 정렬해 이동정비반에 배정하고 출동·정비·복귀를 모의한다.
    baseline: 접수 순서 + 거리 최근접 정비반 / priority: 우선순위 점수 순서 + 도착 예상시각·당일 부하 균형"""
    w = w or W0
    est = np.asarray(est if est is not None else req["std_repair_h"].values, float)
    n_days = int(req["day"].max()) + 1
    free = {t["team"]: 0.0 for t in teams}
    load = {t["team"]: np.zeros(n_days + 5) for t in teams}
    recs, out, pending, i, now = req.to_dict("records"), [], [], 0, cycle_h
    while i < len(recs) or pending:
        while i < len(recs) and recs[i]["t_req"] <= now:
            pending.append((i, recs[i])); i += 1
        if policy == "priority":
            pending.sort(key=lambda p: -priority(p[1], now, est[p[0]], w))
        for k, r in pending:
            cand = [t for t in teams if r["equip"] in t["skills"]]
            day = min(int(now // 24), n_days + 4)
            if policy == "baseline":
                tm = min(cand, key=lambda t: np.hypot(t["x"] - r["x"], t["y"] - r["y"]))
            else:
                tm = min(cand, key=lambda t: max(free[t["team"]], now) + travel_h(t, r)
                         + w["balance"] * load[t["team"]][day] / DUTY_H * 4)
            backlog = max(0.0, free[tm["team"]] - now)
            go = max(free[tm["team"]], now)
            tr = travel_h(tm, r, actual=True)
            work = on_site_h(tm, r)
            free[tm["team"]] = go + 2 * tr + work
            load[tm["team"]][min(int(go // 24), n_days + 4)] += 2 * tr + work
            out.append({**r, "idx": k, "team": tm["team"], "dist_km": float(np.hypot(tm["x"] - r["x"], tm["y"] - r["y"])),
                        "backlog_h": backlog, "cycle_wait_h": now - r["t_req"], "dispatch": go, "travel_h": tr,
                        "work_h": work, "response_h": go + tr - r["t_req"], "downtime_h": go + tr + work - r["t_req"]})
        pending, now = [], now + cycle_h
    L = pd.DataFrame({k: v[:n_days] for k, v in load.items()}) / DUTY_H      # 정비반별 일일 부하율
    return pd.DataFrame(out).sort_values("idx").reset_index(drop=True), L


def metrics(res, L):
    c3 = res[res["crit"] == 3]
    return {"평균 도착(h)": round(res["response_h"].mean(), 2), "핵심장비 도착(h)": round(c3["response_h"].mean(), 2),
            "핵심장비 90%(h)": round(c3["response_h"].quantile(0.9), 2), "평균 부하율": round(float(L.values.mean()), 3),
            "부하율 편차": round(float(L.mean().std()), 3), "과부하 일수": int((L > 1.0).values.sum()),
            "총 이동(h)": round(float(res["travel_h"].sum() * 2), 1)}

CW = {1: 1.0, 2: 3.0, 3: 6.0}          # [교육용가상자료] 중요도별 가동중단 1시간 비용 지수
SLOTS = 3                               # 정비반당 한 배정 주기에 받을 수 있는 최대 건수
APPROVE_DELAY = 0.25                    # 승인 절차로 늦어지는 출동 시각(h)


class Audit:
    def __init__(self):
        self.log = []

    def add(self, **body):
        prev = self.log[-1]["hash"] if self.log else "GENESIS"
        body = {"seq": len(self.log), **body}
        self.log.append({**body, "prev": prev, "hash": hashlib.sha256((prev + json.dumps(body, sort_keys=True, ensure_ascii=False)).encode()).hexdigest()})

    def verify(self):
        prev = "GENESIS"
        for e in self.log:
            b = {k: v for k, v in e.items() if k not in ("prev", "hash")}
            if e["prev"] != prev or hashlib.sha256((prev + json.dumps(b, sort_keys=True, ensure_ascii=False)).encode()).hexdigest() != e["hash"]:
                return e["seq"]
            prev = e["hash"]
        return None


def default_plan(pending, teams, free, now):
    """기본 계획: 접수 순서대로 최근접(정비 가능) 정비반."""
    plan, f = [], dict(free)
    for k, r in pending:
        tm = min([t for t in teams if r["equip"] in t["skills"]], key=lambda t: np.hypot(t["x"] - r["x"], t["y"] - r["y"]))
        go = max(f[tm["team"]], now); plan.append((k, tm)); f[tm["team"]] = go + 2 * travel_h(tm, r) + on_site_h(tm, r)
    return plan


def mip_plan(pending, teams, free, now):
    """할당 문제(정수계획): 요청 x (정비반, 순번) 비용 = 중요도 가중 x 예상 도착시각. 정비 불가 조합은 큰 비용."""
    avg = np.mean([on_site_h(teams[0], r) for _, r in pending]) + 1.0
    cols = [(t, s) for t in teams for s in range(SLOTS)]
    cost = np.full((len(pending), len(cols)), 1e6)
    for i, (k, r) in enumerate(pending):
        for j, (t, s) in enumerate(cols):
            if r["equip"] in t["skills"]:
                cost[i, j] = CW[r["crit"]] * (max(free[t["team"]], now) + s * avg + travel_h(t, r) - now)
    ri, cj = linear_sum_assignment(cost)
    order = sorted(zip(ri, cj), key=lambda p: (cols[p[1]][1], -pending[p[0]][1]["crit"]))
    return [(pending[i][0], cols[j][0]) for i, j in order]


def expected_cost(plan, recs, free, now, delay=0.0):
    f, c = dict(free), 0.0
    for k, tm in plan:
        r = recs[k]; go = max(f[tm["team"]], now + delay)
        c += CW[r["crit"]] * (go + travel_h(tm, r) + on_site_h(tm, r) - r["t_req"]); f[tm["team"]] = go + 2 * travel_h(tm, r) + on_site_h(tm, r)
    return c


def commander(prop, rng):
    """[모의 정비대대장] 기대 이득이 작거나 담당 구역을 크게 벗어나는 재배치는 반려, 10%는 30분 내 무응답."""
    if rng.random() < 0.10:
        return "무응답", "30분 내 응답 없음 -> 기본 계획 유지"
    if prop["max_km"] > 45:
        return "반려", f"정비반 구역 이탈 {prop['max_km']:.0f}km(복귀·보급 곤란)"
    if prop["gain"] < 1.0:
        return "반려", f"기대 이득 {prop['gain']:.1f} 작음 -> 현 계획 유지"
    return "승인", f"핵심장비 우선, 기대 이득 {prop['gain']:.1f}"


def run(req, teams, mode, seed=0, audit=None):
    """mode: default(기본 계획) / hitl(제안 -> 승인 시 적용) / auto(모든 제안 적용: 비교용 상한, 실제 운용 금지)"""
    rng = np.random.default_rng(seed)
    recs = req.to_dict("records"); free = {t["team"]: 0.0 for t in teams}
    out, i, now, props = [], 0, 1.0, []
    while i < len(recs):
        pending = []
        while i < len(recs) and recs[i]["t_req"] <= now:
            pending.append((i, recs[i])); i += 1
        if pending:
            base = default_plan(pending, teams, free, now)
            plan, delay = base, 0.0
            if mode != "default":
                prop = mip_plan(pending, teams, free, now)
                if [(k, t["team"]) for k, t in prop] != [(k, t["team"]) for k, t in base]:
                    gain = expected_cost(base, recs, free, now) - expected_cost(prop, recs, free, now, APPROVE_DELAY)
                    max_km = max(np.hypot(t["x"] - recs[k]["x"], t["y"] - recs[k]["y"]) for k, t in prop)
                    info = {"t": round(now, 2), "n": len(pending), "gain": round(gain, 2), "max_km": max_km}
                    if gain <= 0:                       # 이득 없는 변경은 제안하지 않는다
                        pass
                    elif mode == "auto":
                        dec, why = "승인", "비교용 전부 적용"
                    else:
                        dec, why = commander(info, rng)
                        audit.add(event="PROPOSE", t=info["t"], n=info["n"], gain=info["gain"], status="PENDING")
                        audit.add(event="DECIDE", t=info["t"], decision=dec, reason=why, by="정비대대장(모의)")
                    if gain > 0:
                        props.append({**info, "decision": dec})
                    if gain > 0 and dec == "승인":
                        plan, delay = prop, APPROVE_DELAY
            for k, tm in plan:
                r = recs[k]; go = max(free[tm["team"]], now + delay)
                tr, work = travel_h(tm, r, actual=True), on_site_h(tm, r)
                free[tm["team"]] = go + 2 * tr + work
                out.append({**r, "team": tm["team"], "response_h": go + tr - r["t_req"], "downtime_h": go + tr + work - r["t_req"]})
        now += 1.0
    res = pd.DataFrame(out)
    res["cost"] = res["crit"].map(CW) * res["downtime_h"]
    return res, pd.DataFrame(props)


teams = make_teams()
req = make_requests(seed=1)
audit = Audit()
R = {}
for mode in ("default", "hitl", "auto"):
    R[mode] = run(req, teams, mode, seed=3, audit=audit)
rows = []
for mode, name in [("default", "기본 계획"), ("hitl", "제안 + 대대장 승인"), ("auto", "제안 전부 적용(비교용)")]:
    res = R[mode][0]
    rows.append({"방식": name, "가동중단 비용지수": round(res["cost"].sum()), "핵심장비 도착(h)": round(res.loc[res["crit"] == 3, "response_h"].mean(), 2),
                 "핵심장비 가동중단(h)": round(res.loc[res["crit"] == 3, "downtime_h"].mean(), 2), "전체 도착(h)": round(res["response_h"].mean(), 2)})
T = pd.DataFrame(rows)
print("[교육용가상자료] 30일 모의 결과")
print(T.to_string(index=False))
cut = 1 - T.loc[1, "가동중단 비용지수"] / T.loc[0, "가동중단 비용지수"]
print(f"HITL 적용 가동중단 비용 절감률 {cut:.1%} (목표 15% 이상: {'달성' if cut >= 0.15 else '미달'})")

P = R["hitl"][1]
print(f"\n변경 제안 {len(P)}건: " + ", ".join(f"{k} {v}건" for k, v in P["decision"].value_counts().items()))
rej = [e["reason"] for e in audit.log if e.get("decision") == "반려"]
print("반려 사유 예:", "; ".join([next(x for x in rej if "구역" in x), next(x for x in rej if "이득" in x)]))
n_dec = sum(e["event"] == "DECIDE" for e in audit.log)
print(f"제안 {len(P)}건 중 결정 기록 {n_dec}건(로깅률 {n_dec / max(len(P), 1):.0%}), 자동 승인 0건, 무응답 시 기본 계획 유지")
print(f"감사 로그 무결성: {'정상' if audit.verify() is None else '이상'}", end=", ")
audit.log[5]["reason"] = "사후 수정"
print(f"사후 변조 시 seq {audit.verify()}에서 탐지")

# 민감도: 승인 지연이 길어지면 이득이 줄어든다(전시 전환 등 급증 대비)
print("\n[민감도] 승인 지연(h) -> 비용 절감률")
for d in (0.0, 0.25, 0.5, 1.0):
    APPROVE_DELAY = d
    r_, _ = run(req, teams, "hitl", seed=3, audit=Audit())
    print(f"  {d:>4}: {1 - r_['cost'].sum() / T.loc[0, '가동중단 비용지수']:.1%}")
surge = make_requests(seed=1, per_day=16)
APPROVE_DELAY = 0.25
b, _ = run(surge, teams, "default"); h, _ = run(surge, teams, "hitl", seed=3, audit=Audit())
print(f"요청 1.6배 급증 시나리오: 비용 절감률 {1 - h['cost'].sum() / b['cost'].sum():.1%}")

plt.figure(figsize=(7, 4))
for mode, lab in [("default", "default plan"), ("hitl", "proposal + approval")]:
    s = R[mode][0]
    plt.hist(s.loc[s["crit"] == 3, "response_h"], bins=20, alpha=0.6, label=lab)
plt.xlabel("critical equipment response time (h)"); plt.ylabel("count"); plt.legend(); plt.title("HITL simulator: critical response")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 07 예상 실행 결과 (Output)</b></summary>

```text
[교육용가상자료] 30일 모의 결과
           방식  가동중단 비용지수  핵심장비 도착(h)  핵심장비 가동중단(h)  전체 도착(h)
        기본 계획       6495        3.13          5.82      3.37
  제안 + 대대장 승인       5254        2.10          4.71      2.29
제안 전부 적용(비교용)       5018        1.92          4.53      2.08
HITL 적용 가동중단 비용 절감률 19.1% (목표 15% 이상: 달성)

변경 제안 103건: 승인 71건, 반려 24건, 무응답 8건
반려 사유 예: 정비반 구역 이탈 50km(복귀·보급 곤란); 기대 이득 0.5 작음 -> 현 계획 유지
제안 103건 중 결정 기록 103건(로깅률 100%), 자동 승인 0건, 무응답 시 기본 계획 유지
감사 로그 무결성: 정상, 사후 변조 시 seq 5에서 탐지

[민감도] 승인 지연(h) -> 비용 절감률
   0.0: 20.0%
  0.25: 19.1%
   0.5: 17.6%
   1.0: 13.4%
요청 1.6배 급증 시나리오: 비용 절감률 46.5%
```
</details>
